In [ ]:
'''This notebook entails whether a particular set of Romanized Kannada words is capable of undergoing Script Induced Semantic Hallucination or not in the google/gemma-2-9b-it model.'''

In [1]:
!pip -q install -U transformers accelerate sentencepiece
#Sentencepiece: Tokenizer used by gemma & qwen
#transformers: Imports the tokenizer and model

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 150.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.5 MB/s eta 0:00:00


In [4]:
!pip -q install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 20.5 MB/s eta 0:00:00


In [5]:
from huggingface_hub import notebook_login
notebook_login()

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-2-9b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    #Imports the actual model
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 3584, padding_idx=0)
    (layers): ModuleList(
      (0-41): 42 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=3584, out_features=4096, bias=False)
          (k_proj): Linear(in_features=3584, out_features=2048, bias=False)
          (v_proj): Linear(in_features=3584, out_features=2048, bias=False)
          (o_proj): Linear(in_features=4096, out_features=3584, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=3584, out_features=14336, bias=False)
          (up_proj): Linear(in_features=3584, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=3584, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (pre_feedforward_

In [7]:
def make_prefix(user_prompt):
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [8]:
@torch.no_grad()
def completion_logprob(prompt, completion):
    prefix = make_prefix(prompt)

    # Tokenize the prompt.
    prefix_ids = tokenizer(
        prefix,
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(model.device)

    # Leading space is usually appropriate for an answer continuation.
    candidate_text = " " + completion.strip()

    candidate_ids = tokenizer(
        candidate_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(model.device)

    # Construct:
    # [prompt tokens][candidate tokens]
    input_ids = torch.cat(
        [prefix_ids, candidate_ids],
        dim=1
    )

    outputs = model(input_ids=input_ids)

    logits = outputs.logits

    log_probs = torch.log_softmax(
        logits[:, :-1, :].float(),
        dim=-1
    )

    targets = input_ids[:, 1:]

    token_log_probs = log_probs.gather(
        2,
        targets.unsqueeze(-1)
    ).squeeze(-1)

    # Candidate begins here in target indexing.
    start = prefix_ids.shape[1] - 1
    end = start + candidate_ids.shape[1]

    candidate_log_probs = token_log_probs[:, start:end]

    return {
        "sum_logprob": candidate_log_probs.sum().item(),
        "mean_logprob": candidate_log_probs.mean().item(),
        "num_tokens": candidate_ids.shape[1],
        "tokens": tokenizer.convert_ids_to_tokens(
            candidate_ids[0]
        )
    }

In [9]:
prompt = """
What is the English meaning of the highlighted word?

Sentence: avanu baki kelasa mugisida
Word: baki

Answer only with the meaning.
"""

correct = "remaining"
english_attractor = "back"

In [10]:
K = completion_logprob(prompt, correct)
E = completion_logprob(prompt, english_attractor)

print("Correct Kannada meaning:")
print(K)

print("\nEnglish attractor:")
print(E)

Correct Kannada meaning:
{'sum_logprob': -9.613738059997559, 'mean_logprob': -9.613738059997559, 'num_tokens': 1, 'tokens': ['▁remaining']}

English attractor:
{'sum_logprob': -30.019987106323242, 'mean_logprob': -30.019987106323242, 'num_tokens': 1, 'tokens': ['▁back']}


In [11]:
M_sish = E["mean_logprob"] - K["mean_logprob"]

print("\nM_SISH =", M_sish)


M_SISH = -20.406249046325684


In [12]:
prompt_roman = """
What is the English meaning of the highlighted word?

Sentence: avanu baki kelasa mugisalilla
Word: baki

Answer only with the meaning.
"""

prompt_native = """
What is the English meaning of the highlighted word?

Sentence: ಅವನು ಬಾಕಿ ಕೆಲಸ ಮುಗಿಸಲಿಲ್ಲ
Word: ಬಾಕಿ

Answer only with the meaning.
"""

In [13]:
def sish_margin(prompt, correct, attractor):
    K = completion_logprob(prompt, correct)
    E = completion_logprob(prompt, attractor)

    return {
        "correct_logp": K["mean_logprob"],
        "attractor_logp": E["mean_logprob"],
        "margin": E["mean_logprob"] - K["mean_logprob"]
    }


roman = sish_margin(
    prompt_roman,
    correct,
    english_attractor
)

native = sish_margin(
    prompt_native,
    correct,
    english_attractor
)

print("Romanized:", roman)
print("Native:", native)

delta_script = roman["margin"] - native["margin"]

print("Δ_script =", delta_script)

Romanized: {'correct_logp': -9.01478099822998, 'attractor_logp': -24.655405044555664, 'margin': -15.640624046325684}
Native: {'correct_logp': -10.90051555633545, 'attractor_logp': -38.900516510009766, 'margin': -28.000000953674316}
Δ_script = 12.359376907348633
